Imports Python libraries

In [1]:
import pandas as pd

Loads data from insurance dataset

In [2]:
df = pd.read_csv('../data/insurance_data.csv')
df.head()

/var/folders/bk/jmg3rr0j58s3yrt8fdrdc1ph0000gn/T/ipykernel_97576/3968581275.py:1: DtypeWarning: Columns (0: COMPLEX_NA) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../data/insurance_data.csv')


,Year,ZIP,Avg Fire Risk Score,Avg PPC,CAT Cov A Fire - Incurred Losses,CAT Cov A Fire - Number of Claims,CAT Cov A Smoke - Incurred Losses,CAT Cov A Smoke - Number of Claims,CAT Cov C Fire - Incurred Losses,CAT Cov C Fire - Number of Claims,...,average_household_size,educational_attainment_bachelor_or_higher,poverty_status,housing_occupancy_number,housing_value,year_structure_built,housing_vacancy_number,median_monthly_housing_costs,owner_occupied_housing_units,renter_occupied_housing_units
0,2018,90003,1.000,1.000,0,0,0,0,0,0,...,4.03,2294.0,18743.0,18349.0,547600.0,18349.0,715.0,1609.0,4692.0,12942.0
1,2018,90004,0.420,1.075,0,0,0,0,0,0,...,2.44,11822.0,10994.0,26046.0,1457200.0,26046.0,2311.0,1847.0,4011.0,19724.0
2,2018,90005,0.510,1.060,0,0,0,0,0,0,...,2.18,7582.0,9463.0,19357.0,1084400.0,19357.0,1880.0,1651.0,1572.0,15905.0
3,2018,90006,0.605,1.040,0,0,0,0,0,0,...,2.79,6758.0,13274.0,21475.0,841900.0,21475.0,2207.0,1415.0,1951.0,17317.0
4,2018,90007,0.000,2.000,0,0,0,0,0,0,...,2.75,3536.0,13473.0,14153.0,852900.0,14153.0,1702.0,1504.0,1297.0,11154.0


Creates zip_year dataset, which condenses data within unique zip code + year combos; totals are summed together, constants are retained, and averages are further averaged together.

In [3]:
zip_year = df.groupby(["ZIP", "Year"], as_index=False).apply(
    lambda g: pd.Series({
        "Earned Premium": g["Earned Premium"].sum(),
        "Earned Exposure": g["Earned Exposure"].sum(),

        "Avg Fire Risk Score": ((g["Avg Fire Risk Score"] * g["Earned Exposure"]).sum() / g["Earned Exposure"].sum()
            if g["Earned Exposure"].sum() != 0 else None ),
        "Avg PPC": ((g["Avg PPC"] * g["Earned Exposure"]).sum() / g["Earned Exposure"].sum()
            if g["Earned Exposure"].sum() != 0 else None ),
        
        "Cov A Amount Weighted Avg" : (g["Cov A Amount Weighted Avg"].mean()),
        "Cov C Amount Weighted Avg" : (g["Cov C Amount Weighted Avg"].mean()),
        
        "Number of Very High Fire Risk Exposure": g["Number of Very High Fire Risk Exposure"].sum(),
        "Number of High Fire Risk Exposure": g["Number of High Fire Risk Exposure"].sum(),
        "Number of Moderate Fire Risk Exposure": g["Number of Moderate Fire Risk Exposure"].sum(),
        "Number of Low Fire Risk Exposure": g["Number of Low Fire Risk Exposure"].sum(),
        "Number of Negligible Fire Risk Exposure": g["Number of Negligible Fire Risk Exposure"].sum(),
        
        "median_income": g["median_income"].dropna().iloc[0] if g["median_income"].notna().any() else None,
        "total_population": g["total_population"].dropna().iloc[0] if g["total_population"].notna().any() else None,
        "total_housing_units": g["total_housing_units"].dropna().iloc[0] if g["total_housing_units"].notna().any() else None,
        "Poverty Rate": g["poverty_status"].iloc[0] / g["total_population"].iloc[0] if g["total_population"].iloc[0] != 0 else None
    })
).reset_index(drop=True)

Creates column to learn from last year's premium and drops columns without that ability to learn from past data

In [4]:
zip_year['last_year_premium'] = zip_year.groupby('ZIP')['Earned Premium'].shift(1)

In [5]:
zip_year = zip_year.dropna(subset=["last_year_premium"])


Instantiates features and targets for training

In [6]:
features = ["Earned Exposure", "Avg Fire Risk Score", "Avg PPC", "Cov A Amount Weighted Avg", "Cov C Amount Weighted Avg",
            "Number of Very High Fire Risk Exposure", "Number of High Fire Risk Exposure", "Number of Moderate Fire Risk Exposure", "Number of Low Fire Risk Exposure", "Number of Negligible Fire Risk Exposure",
            "median_income", "total_population", "total_housing_units", "Poverty Rate", "last_year_premium"]
target = "Earned Premium"

train_zy = zip_year[zip_year["Year"] < 2021]
test_zy = zip_year[zip_year["Year"] == 2021]

X_train = train_zy[features].copy()
X_test = test_zy[features].copy()
y_train = train_zy[target].copy()
y_test = test_zy[target].copy()

Uses median to fill in missing values

In [7]:
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='median')
X_train_imputed = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_imputed = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns, index=X_test.index)

Feature engineering, log transforms, and imputation, followed by gradient boosting

In [8]:
from sklearn.ensemble import GradientBoostingRegressor
import numpy as np

X_train_imputed["Risk Exposure"] = (X_train_imputed["Avg Fire Risk Score"] * X_train_imputed["Earned Exposure"])
X_test_imputed["Risk Exposure"] = (X_test_imputed["Avg Fire Risk Score"] * X_test_imputed["Earned Exposure"])

X_train_imputed["log exposure"] = np.log1p(np.clip(X_train_imputed["Earned Exposure"], 0, None))
X_test_imputed["log exposure"] = np.log1p(np.clip(X_test_imputed["Earned Exposure"], 0, None))

X_train_imputed["log last year premium"] = np.log1p(np.clip(X_train_imputed["last_year_premium"], 0, None))
X_test_imputed["log last year premium"] = np.log1p(np.clip(X_test_imputed["last_year_premium"], 0, None))
risk_cols = ["Number of Very High Fire Risk Exposure", "Number of High Fire Risk Exposure", "Number of Moderate Fire Risk Exposure", "Number of Low Fire Risk Exposure", "Number of Negligible Fire Risk Exposure"]

train_total_risk = X_train_imputed[risk_cols].sum(axis=1)
test_total_risk = X_test_imputed[risk_cols].sum(axis=1)

X_train_imputed = X_train_imputed.replace([np.inf, -np.inf], np.nan)
X_test_imputed = X_test_imputed.replace([np.inf, -np.inf], np.nan)

X_train_imputed = X_train_imputed.fillna(X_train_imputed.median())
X_test_imputed = X_test_imputed.fillna(X_test_imputed.median())

model = GradientBoostingRegressor(n_estimators=500, learning_rate=0.025, max_depth=3, min_samples_leaf=15, random_state=42)

upper = y_train.quantile(0.995)
y_train_clipped = np.clip(y_train, 0, upper)
y_train_log = np.log1p(y_train_clipped)

model.fit(X_train_imputed, y_train_log)
y_pred = np.expm1(model.predict(X_test_imputed))

Analysis Metrics

In [9]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

mae = mean_absolute_error(y_test, y_pred)
rmse = root_mean_squared_error(y_test, y_pred)
mask = y_test >= 100
y_test_safe = np.clip(y_test, 1, None)
pct_error = np.abs(y_test - y_pred) / y_test_safe
mape = np.mean(pct_error) * 100
mape_100 = np.mean(np.abs(y_test[mask] - y_pred[mask]) / y_test[mask]) * 100
within_20 = np.mean(pct_error < 0.20) *100
within_5 = np.mean(pct_error < 0.05) *100

print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"MAPE: {mape:.2f}%")
print(f"MAPE >= 100: {mape_100:.2f}%")
print(f"Within 20%: {within_20:.2f}%")
print(f"Within 5%: {within_5:.2f}%")

MAE: 655230.87
RMSE: 2252640.01
MAPE: 71.49%
MAPE >= 100: 25.43%
Within 20%: 54.77%
Within 5%: 13.41%
